In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.products"
)

silver_df = spark.table(
    "customer360_dev.silver.products"
)

reject_df = spark.table(
    "customer360_dev.quarantine.product_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 150,024
Silver rows   : 149,400
Rejected rows : 624


Rebuild the critical validation logic

In [0]:
validated_df = (
    bronze_df
    .withColumn(
        "unit_price_cast",
        F.col("unit_price").cast("decimal(12,2)")
    )
    .withColumn(
        "cost_price_cast",
        F.col("cost_price").cast("decimal(12,2)")
    )
    .withColumn(
        "launch_date_cast",
        F.to_date("launch_date", "yyyy-MM-dd")
    )
    .withColumn(
        "discontinued_date_cast",
        F.to_date("discontinued_date", "yyyy-MM-dd")
    )
    .withColumn(
        "product_name_clean",
        F.trim("product_name")
    )
    .withColumn(
        "product_status_clean",
        F.upper(F.trim("product_status"))
    )
    .withColumn(
        "currency_code_clean",
        F.upper(F.trim("currency_code"))
    )
    .withColumn(
        "dq_missing_product_id",
        F.col("product_id").isNull()
        | (F.trim(F.col("product_id")) == "")
    )
    .withColumn(
        "dq_missing_product_name",
        F.col("product_name_clean").isNull()
        | (F.col("product_name_clean") == "")
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("product_status_clean").isin(
            "ACTIVE",
            "INACTIVE",
            "DISCONTINUED"
        )
    )
    .withColumn(
        "dq_invalid_unit_price",
        F.col("unit_price_cast").isNull()
        | (F.col("unit_price_cast") < 0)
    )
    .withColumn(
        "dq_invalid_cost_price",
        F.col("cost_price_cast").isNull()
        | (F.col("cost_price_cast") < 0)
    )
    .withColumn(
        "dq_invalid_date_range",
        F.col("launch_date_cast").isNull()
        | (
            F.col("discontinued_date_cast").isNotNull()
            & (
                F.col("discontinued_date_cast")
                < F.col("launch_date_cast")
            )
        )
    )
    .withColumn(
        "dq_invalid_currency",
        ~F.col("currency_code_clean").isin(
            "INR",
            "USD"
        )
    )
    .withColumn(
        "dq_critical_failure",
        F.col("dq_missing_product_id")
        | F.col("dq_missing_product_name")
        | F.col("dq_invalid_status")
        | F.col("dq_invalid_unit_price")
        | F.col("dq_invalid_cost_price")
        | F.col("dq_invalid_date_range")
        | F.col("dq_invalid_currency")
    )
)

Count valid candidates

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

Valid candidates before dedup: 149,400


Calculate duplicates removed

In [0]:
duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Duplicate rows removed: 0


Reconcile totals

In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 150,024
Silver rows        : 149,400
Rejected rows      : 624
Duplicates removed : 0
Accounted for      : 150,024
Difference         : 0


Create reconciliation result

In [0]:
result = [
    {
        "entity": "products",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
150024,150024,0,0,products,624,149400,PASS


Analyze rejection reasons

In [0]:
(
    reject_df
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+--------------------------------------------------------------------------------+-----+
|rejection_reason                                                                |count|
+--------------------------------------------------------------------------------+-----+
|INVALID_PRODUCT_STATUS                                                          |300  |
|INVALID_UNIT_PRICE                                                              |150  |
|INVALID_DATE_RANGE                                                              |150  |
|MISSING_PRODUCT_NAME; INVALID_UNIT_PRICE; INVALID_COST_PRICE; INVALID_DATE_RANGE|24   |
+--------------------------------------------------------------------------------+-----+



Measure warning metrics

In [0]:
warning_summary = (
    silver_df
    .select(
        F.sum(
            F.when(
                F.col("dq_warning_selling_below_cost") == True,
                1
            ).otherwise(0)
        ).alias("selling_below_cost"),

        F.sum(
            F.when(
                F.col("dq_warning_discontinued_without_date") == True,
                1
            ).otherwise(0)
        ).alias("discontinued_without_date")
    )
)

display(warning_summary)

selling_below_cost,discontinued_without_date
0,0


Validate product-key uniqueness

In [0]:
duplicate_product_keys = (
    silver_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate product IDs in Silver:",
    duplicate_product_keys
)

Duplicate product IDs in Silver: 0
